# CNNs and ResNets

## PyTorch `nn.Module`

First we will start with the basics of creating different models.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchinfo
import einops

from torch import Tensor
from jaxtyping import Float, Int
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

### The `nn.Parameter` class

`nn.Parameter` is a special type of `Tensor` object. It's, at its core, the class that PyTorch has provided for storing the weights and biases of a `Module`.

* If a `Parameter` is set as an attribute of a `Module`, it will be detected by PyTorch automatically and returned when calling `module.parameters()`.
* This makes it easy to pass all parameters of a model into an optimizer and update them at all once. 

We can also printing information with `extra_repr`:

`extra_repr` allows us to format the string representation of our `Module` in a more informative way than the default.


```python
class MyModule(nn.Module):
    def __init__(self, arg1, arg2, ...):
        super().__init__()

    def extra_repr(self) -> str:
        return f"arg1={self.arg1}, arg2={self.arg2}, ..."
```

### Implementing ReLU

ReLU takes in a `Tensor` object and returns the elements of the `Tensor` if it is greater than 0, otherwise, it replaces those elements with 0.


$$ReLU = \max \{0, t\}$$

In [2]:
class ReLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return torch.maximum(torch.tensor([0.]), x)

relu = ReLU()
x = torch.randn(5)
print(f"{x=}")
print(f"{relu(x)=}")


x=tensor([ 0.8002, -0.8214,  0.9027,  1.0107,  0.0187])
relu(x)=tensor([0.8002, 0.0000, 0.9027, 1.0107, 0.0187])


### Implementing Linear Module

Based on the PyTorch documentation found [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).

Based on this documentation the `Linear` module applies an affine linear transformation:

$$
y = XW^{T} + b
$$

__Parameters__:
* __in_features__ (_int_) - size of each input sample
* __out_features__ (_int_) - size of each input sample
* __bias__ (_bool_) - if set to `False`, layer will not learn an additive bias

__Shape__:
* Input `(*, H_in)` where `*` means any number of dimension including `None`.
* Output `(*, H_in)` where all but the last dimension are the same shape as the input and $H_{out}=\textbf{out\_features}$ 


For the weights - implement the _Kaiming Initialization_, where each float in the weight and bias tensors are drawn independently from the uniform distrubtion on the interval:

$$
\left[ -\frac{1}{\sqrt{N_{in}}},\frac{1}{\sqrt{N_{in}}}\right]
$$


More info on initialization strategies and why it's important can be found [here](https://pouannes.github.io/blog/initialization/) and [here](https://adityassrana.github.io/blog/theory/2020/08/26/Weight-Init.html)

In [ ]:
import math

class Linear(nn.Module):
    def __init__(self, in_features:int, out_features:int, bias:bool=True):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        scaling_factor = math.sqrt(in_features)
        weights = (2*torch.rand(out_features, in_features)) - 1.

        # self.weight and self.bias are how PyTorch names their attributes so will we so that we can compare again the official PyTorch modules
        self.weight = nn.Parameter(weights/scaling_factor)

        if bias:
            b = (2*torch.rand(out_features)) - 1.
            self.bias = nn.Parameter(b/scaling_factor)
        else:
            self.bias = None

    def forward(self, x:Tensor) -> Tensor:

        out = einops.einsum(x, self.weight, "... in_feats, ... out_feats in_feats -> ... out_feats")

        if self.bias is not None:
            out += self.bias

        return out

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"

In [14]:
in_features = 784
out_features = 10

model = Linear(in_features, out_features, bias=True)
print(model)

Linear(in_features=784, out_features=10, bias=True)


In [15]:
def test_linear_forward(Linear, bias=False):
    x = torch.rand((10, 512))
    our_model = Linear(512, 64, bias=bias)
    official = torch.nn.Linear(512, 64, bias=bias)

    our_model.load_state_dict(official.state_dict())
    actual = our_model(x)
    expected = official(x)
    torch.testing.assert_close(actual, expected)
    print("All tests in `test_linear_forward` passed")

test_linear_forward(Linear)
test_linear_forward(Linear, bias=True)

All tests in `test_linear_forward` passed
All tests in `test_linear_forward` passed


### Flatten Module

This module is grabbed from ARENA's curriculum found [here](https://learn.arena.education/chapter0_fundamentals/02_cnns/1-making-your-own-modules/)

In [17]:
class Flatten(nn.Module):
    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: Tensor) -> Tensor:
        """
        Flatten out dimensions from start_dim to end_dim, inclusive of both.
        """
        shape = input.shape

        # Get start & end dims, handling negative indexing for end dim
        start_dim = self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim

        # Get the shapes to the left / right of flattened dims, as well as size of flattened middle
        shape_left = shape[:start_dim]
        shape_right = shape[end_dim + 1 :]
        shape_middle = torch.prod(torch.tensor(shape[start_dim : end_dim + 1])).item()

        return torch.reshape(input, shape_left + (shape_middle,) + shape_right)

    def extra_repr(self) -> str:
        return ", ".join([f"{key}={getattr(self, key)}" for key in ["start_dim", "end_dim"]]) 